# Неделя 4: Transfer Learning, BERT (домашнее задание)

## Поисковик похожих вопросов

Эмбеддинги — полезный источник информации для многих задач. Например, с их помощью можно классифицировать тексты или находить похожие документы. Мы уже знакомы с Word2Vec, GloVe и fastText, но эти модели не учитывают контекст **конкретного обрабатываемого текста** (они используют лишь контексты обучающего корпуса).

Сегодня воспользуемся всей мощью **контекстных эмбеддингов**, чтобы искать дубликаты текстов!

**Внимание:** предполагается, что ты уже изучил `seminar.ipynb`.

> Неофициальный перевод [оригинального задания](https://github.com/yandexdataschool/nlp_course/blob/2026/week04_transfer/homework.ipynb). Код и автопроверки сохранены.

## Правила выполнения

Этот ноутбук автоматически проверяется системой **ThetaGrader (TG)**.

- **Не меняй заголовки вида `### Task N`** — TG использует их для поиска заданий. Не удаляй, не переименовывай, не перенумеровывай и не дублируй эти заголовки.
- Пиши решение только в предназначенных для него ячейках сразу под соответствующим заданием. Не переносí решения в другие части ноутбука и не объединяй несколько разных решений.
- Если задание требует текстового вывода, отчёта или ответа на теоретический вопрос, запиши его в Markdown-ячейке сразу под заданием отдельно от кода.
- Перед отправкой убедись, что поле `Status` в ячейке Bot-check ниже имеет значение `final`.

## Информация об использовании ИИ

Если при выполнении домашней работы ты использовал ИИ-помощников (например, ChatGPT, Claude, GitHub Copilot), кратко опиши, какие инструменты применял, в каких частях задания и каким образом.

Если не использовал, оставь в этой ячейке `no AI assistance used`.

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Подготовка данных

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

### Оценка качества (1 балл)

Мы выбрали готовую модель, обученную на QQP. Но насколько она хороша?

Один из способов — измерить accuracy на валидационной выборке. Именно это предстоит реализовать дальше.

Ниже приведён интерфейс, который поможет с задачей.

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

### Task 1 (1 point):

**Задание 1 (1 балл)**

Измерь accuracy модели на валидационной выборке. Наивное решение может работать несколько часов, поэтому обязательно используй оптимизации:

- Запускай модель на GPU с `no_grad`.
- Используй размер батча больше 1.
- Оптимизируй загрузку данных через `num_workers > 1`.
- Необязательно: примени [смешанную точность](https://pytorch.org/docs/stable/notes/amp_examples.html).

In [ ]:
<A whole lot of YOUR CODE HERE>

accuracy = <Validation accuracy, between 0 and 1>

In [ ]:
assert 0.9 < accuracy < 0.91

### Обучение (4 балла)

В этом задании можно выбрать один из двух вариантов.

**Вариант A:** дообучи собственную модель (**fine-tuning**). Разрешена любая архитектура, **кроме оригинального BERT**. Рекомендуем [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Ещё лучше — выбери лучшую модель по публичным бенчмаркам, например [GLUE](https://gluebenchmark.com/).

Можно написать цикл обучения самостоятельно либо воспользоваться `transformers.Trainer` (см. [пример](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Убедись, что точность не хуже или хотя бы **сопоставима** с приведённым выше примером BERT.

**Вариант B:** сравни **как минимум три** ранее дообученные модели, не считая приведённого выше BERT. Для каждой укажи:
1. Accuracy.
2. Скорость обработки в примерах в секунду на твоём оборудовании.
3. Размер модели в мегабайтах.

Сравнивай модели в одинаковых условиях — на том же CPU/GPU. Результаты сведи в таблицу и напиши краткий отчёт объёмом примерно половину страницы сверх таблицы.

### Task 2 (4 points):

**Задание 2 (4 балла)**

- Выбери вариант A или B (**оцениваться будет только один**).
- Соблюдай все требования и ограничения выбранного варианта.

In [ ]:
<A whole lot of YOUR CODE HERE>

### Поиск дубликатов (1 балл)

Наконец используем модель для поиска дубликатов вопросов.

Реализуй функцию, которая принимает вопрос и возвращает **пять наиболее вероятных дубликатов** из обучающего набора. Пока не страшно, если она медленная, главное — корректные результаты.

Продемонстрируй работу минимум на пяти примерах.

### Task 3 (1 point):

**Задание 3 (1 балл)**

- Реализуй функцию поиска дубликатов.
- Проверь её минимум на пяти примерах.
- Изучи найденные дубликаты и сделай вывод о корректности работы модели.

In [ ]:
<A whole lot of YOUR CODE HERE>

### Дополнительно: ускорение поиска дубликатов (0,5 балла)

Попробуй ускорить поиск по сравнению с полным перебором всех вопросов. Например, сначала сформируй короткий список кандидатов более дешёвым методом, а затем используй трансформер только для них.

Если выбрал этот вариант, **сохрани и исходную, и оптимизированную реализации**, а также кратко объясни разницу.

### Task 4 (Bonus, 0.5 point):

**Дополнительное задание 4 (0,5 балла)**

- Ускорь реализацию из раздела поиска дубликатов.
- Измерь время исходной и новой версии.
- Опиши свой подход.

In [ ]:
<A whole lot of YOUR CODE HERE>

### Дополнительно: поиск дубликатов классическим способом (1,5 балла)

В этом задании нужно решить задачу поиска дубликатов с помощью предобученных эмбеддингов: Word2Vec, GloVe или fastText.

### Task 5 (Bonus, 1.5 points):

**Дополнительное задание 5 (1,5 балла)**

- Реши поиск дубликатов через предложенные эмбеддинги.
- Сравни классический подход с ранее использованными по качеству, скорости и другим показателям.
- Подготовь короткий отчёт (до пяти шагов, с результатами и выводами).

In [ ]:
<A whole lot of YOUR CODE HERE>